In [19]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage,SystemMessage,BaseMessage
import os
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [20]:
load_dotenv()

True

In [21]:
api_key = os.getenv("groq_api")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing from .env")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    groq_api_key=api_key
)

In [22]:
class chatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [23]:
def ChatBot(state: chatState):
    messages = state["messages"]

    response = llm.invoke(messages)

    return {"messages": [response]}

In [24]:
checkpointer = MemorySaver()
graph = StateGraph(chatState)
graph.add_node('ChatBot',ChatBot)

graph.add_edge(START,'ChatBot')
graph.add_edge('ChatBot',END)

In [25]:
workflow = graph.compile(checkpointer=checkpointer)
initial_state = {"messages": [HumanMessage(content='What is the captial of Pakistan')]}
workflow.invoke(initial_state)

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [30]:
thread_id = '1'



while True:

    userMessage = input('Type here:' )

    print('user:', userMessage)

    if userMessage.strip().lower() in  ['exit','bye','stop']:
        break

    config  = {'configurable': {'thread_id': thread_id}}

    response = workflow.invoke({'messages':[HumanMessage(content=userMessage)]}, config=config)

    print("AI:", response["messages"][-1].content)    


user: hello
AI: Hi again! 👋 How can I help you today?
user: my nme is hashmat 
AI: Got it—your name is Hashmat! How can I help you today?
user: what is m y name 
AI: Your name is Hashmat.
user: what is the meaning of my name 
AI: **Hashmat (حشمت)**

| Language | Script | Pronunciation | Literal Meaning | Cultural/Religious Context |
|----------|--------|---------------|-----------------|----------------------------|
| Arabic | حشمت | *Ḥashmat* | “Pride,” “Dignity,” “Honor” | A virtue highly valued in Islamic ethics; often used in poetry and prose to describe a person’s noble character. |
| Persian | حشمت | *Ḥashmat* | Same as Arabic | Common as a given name in Iran, Afghanistan, and Tajikistan. |
| Urdu | حشمت | *Ḥashmat* | Same as Arabic | Popular in Pakistan and India, especially among Muslim families. |

### Etymology
- The root **ḥ‑š‑m** (ح‑ش‑م) in Semitic languages conveys the idea of “to be proud, to be dignified.”  
- In Arabic, the noun **ḥashmat** (حشمت) literally means “pride